<span></span>
<center>
<h1><b>Fall 2026<br/>
CSC-372 Project Part 8<br/></b></h1>

# Masked Language Modeling

</center>


<a href="https://colab.research.google.com/github/fahadsultan/csc372/blob/main/Project_8.ipynb" target="_blank" align="right">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


## Q0. Install dependencies

In [ ]:
# !pip install torch datasets tiktoken

## Q0. Load and tokenize Tiny Stories

The following code prepares a text dataset for training a Masked Language Model (MLM) using a transformer encoder. It downloads a collection of short stories, tokenizes them using GPT-2's tokenizer, and constructs a smaller vocabulary to make training more computationally efficient.

The code performs five main steps:

<br/>

1. Load the TinyStories dataset

    We use the TinyStories dataset, which contains short, relatively simple stories suitable for training small language models.

    Rather than using the entire dataset, we select 50,000 stories for training and 5,000 for validation. The validation set will allow us to evaluate how well the model performs on stories it has not seen during training.

    An end-of-text token is appended after each story to indicate where one story ends and another begins. This allows us to preserve story boundaries even though the stories are concatenated into a single token sequence.

<br/>

2. Tokenize the stories

    Neural networks operate on numbers rather than raw text. We therefore use GPT-2's tokenizer, which employs byte-pair encoding (BPE), to convert text into sequences of integer token IDs.

    The `tokenize_stories` function processes every story and concatenates its token IDs into one long sequence.

<br/>

3. Construct a smaller vocabulary

    GPT-2's tokenizer has a vocabulary of 50,257 tokens. However, many of these tokens may occur very rarely in our relatively small training dataset.

    We count how frequently each token appears in the training data and retain only tokens that occur at least five times.

    This reduces the number of tokens the model needs to recognize and predict, which in turn reduces the number of parameters in the embedding and output layers.

    We also reserve three special token IDs:


    | Token	| ID	| Purpose |
    |----|---|----|
    | [UNK]	| 0	| Represents tokens excluded from the smaller vocabulary |
    | [MASK] | 1	| Replaces selected tokens during masked language model training |
    | [PAD]	| 2	| Can be used to pad sequences to a common length |

    The retained GPT-2 tokens receive new IDs starting at 3.

<br/>

4. Map between GPT-2 and local token IDs

    We construct two dictionaries: `gpt2_to_local` and `local_to_gpt2`

    The first converts original GPT-2 token IDs into IDs in our smaller vocabulary. The second performs the reverse mapping.

    The `encode` function tokenizes new text and maps the resulting tokens into our vocabulary

    Any token not included in our vocabulary is replaced by [UNK].

    The decode function converts retained local token IDs back into readable text. Special tokens and unknown tokens are omitted because they have no corresponding entry in local_to_gpt2.

    The encode function tokenizes new text and maps the resulting tokens into our vocabulary.

    <br/>

5. Prepare training and validation sequences

    Finally, we convert the tokenized training and validation datasets into the smaller vocabulary:

    We then report the number of tokens, the vocabulary size, and the proportion of validation tokens that became [UNK].

    The validation UNK rate is particularly useful because it measures how often the model encounters tokens outside its vocabulary when processing unseen text.



In [ ]:
import torch
import torch.nn as nn
import tiktoken

from collections import Counter
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = tiktoken.get_encoding("gpt2")

SEQ_LEN = 128
BATCH_SIZE = 32

# A slice of TinyStories is plenty for a small model
train_ds = load_dataset("roneneldan/TinyStories", split="train[:50000]")
valid_ds = load_dataset("roneneldan/TinyStories", split="validation[:5000]")

def tokenize_stories(ds):
    ids = []
    for story in ds["text"]:
        ids.extend(tokenizer.encode_ordinary(story.strip()))
        ids.append(tokenizer.eot_token)   # marks the boundary between stories
    return ids

train_gpt2 = tokenize_stories(train_ds)
valid_gpt2 = tokenize_stories(valid_ds)

# Build a compact vocabulary from GPT-2 tokens that appear at least 5 times.
# Special tokens take the first IDs; PAD never appears in the data.
UNK_ID, MASK_ID, PAD_ID = 0, 1, 2
NUM_SPECIAL = 3

counts = Counter(train_gpt2)
kept = sorted(t for t, c in counts.items() if c >= 5)

gpt2_to_local = {t: i + NUM_SPECIAL for i, t in enumerate(kept)}
local_to_gpt2 = {i: t for t, i in gpt2_to_local.items()}
VOCAB_SIZE = len(kept) + NUM_SPECIAL

def encode(text):
    return [gpt2_to_local.get(t, UNK_ID) for t in tokenizer.encode_ordinary(text)]

def decode(ids):
    return tokenizer.decode([local_to_gpt2[i] for i in ids if i in local_to_gpt2])

train_tokens = [gpt2_to_local.get(t, UNK_ID) for t in train_gpt2]
valid_tokens = [gpt2_to_local.get(t, UNK_ID) for t in valid_gpt2]

print("Training tokens:  ", len(train_tokens))
print("Validation tokens:", len(valid_tokens))
print(f"Vocabulary size:   {VOCAB_SIZE:,} (vs. {tokenizer.n_vocab:,} in GPT-2)")
print(f"Validation UNK rate: {valid_tokens.count(UNK_ID) / len(valid_tokens):.3%}")

Training tokens:   11089438
Validation tokens: 1000238
Vocabulary size:   9,722 (vs. 50,257 in GPT-2)
Validation UNK rate: 0.165%


# Q1. Create a PyTorch Dataset with random masking

Masking happens dynamically in __getitem__, so the same sequence can receive different masks in different epochs.

For simplicity, we'll replace every selected token with [MASK], rather than using BERT's 80/10/10 masking strategy.

---

In this exercise, you will implement a custom PyTorch `Dataset` that prepares training examples for a **Masked Language Model (MLM)**.

<font color="violet">Recall that masked language modeling involves randomly hiding some tokens in a sequence and training a model to predict the original tokens using their surrounding context.</font>

### Task 1: Implement `MaskedDataset`

Create a class called `MaskedDataset` that inherits from `torch.utils.data.Dataset`. Your class should implement the following methods:

**1. `__init__(self, tokens, seq_len=128, seed=None)`**

- Accept a list of token IDs (`tokens`).
- Accept the number of tokens in each training example (`seq_len`).
- Accept an optional random seed (`seed`) for reproducibility.
- Store these values as instance attributes.

**2. `__len__(self)`**

- Return the number of complete, non-overlapping sequences of length `seq_len` that can be constructed from the token list.
- Ignore any leftover tokens that do not form a complete sequence.

**3. `__getitem__(self, idx)`**

Given an index `idx`, perform the following steps:

1. Extract the corresponding sequence of `seq_len` consecutive tokens and convert it into a PyTorch tensor of type `torch.long`.
2. Create a `labels` tensor of the same shape, initialized entirely to `-100`.
3. Randomly select approximately **15% of token positions** for masking.
4. Ensure that at least one token position is selected, even if the random selection produces no masked positions.
5. For every selected position:
   - Copy the original token ID into the corresponding position of `labels`.
   - Replace the token in the input sequence with `MASK_ID`.
6. Return a tuple `(x, labels)` containing the masked input sequence and its labels.

If a seed is provided, use `torch.Generator` with `seed + idx` to make the masking reproducible for each example. Otherwise, allow the mask to change between accesses.

In [ ]:
class MaskedDataset(Dataset):
    def __init__(self, tokens, seq_len=128, seed=None):
        pass

    def __len__(self):
        pass

    def __getitem__(self, idx):
        pass

# Q2. Create DataLoaders

Using your `MaskedDataset` implementation, create two PyTorch `DataLoader` objects:

- **`train_loader`**: Use `train_tokens`, `SEQ_LEN`, and `BATCH_SIZE`. Enable shuffling and allow masking to vary between accesses.
- **`val_loader`**: Use `valid_tokens`, `SEQ_LEN`, and `BATCH_SIZE`. Disable shuffling and use `seed=0` so that validation examples have consistent masks across evaluations.

In [ ]:

# train_loader = 
# val_loader = 

Verify Your Implementation

Retrieve one batch from `train_loader` and verify that:

- Both the input and label tensors have shape `(BATCH_SIZE, SEQ_LEN)` for a full batch.
- Approximately 15% of input positions contain `MASK_ID`.
- Every masked position has its original token ID stored in `labels`.
- Every unmasked position has a label of `-100`.


# Q3. Single Head Attention


In this exercise, you will implement **single-head self-attention from scratch** using PyTorch. Self-attention allows each token in a sequence to incorporate information from other tokens based on their relevance.

Your implementation should use basic PyTorch operations rather than the built-in `nn.MultiheadAttention` module.

<br/>

Create a class called `SingleHeadAttention` that inherits from `nn.Module`.

**1. `__init__(self, d_model=128)`**

Initialize three learnable linear transformations:

- `W_q`: Projects input embeddings into query vectors.
- `W_k`: Projects input embeddings into key vectors.
- `W_v`: Projects input embeddings into value vectors.

Each transformation should map from `d_model` dimensions to `d_model` dimensions. Set `bias=False` for all three layers.

**2. `forward(self, x, padding_mask=None)`**

The input `x` has shape `(batch_size, seq_len, d_model)`. Implement the following steps:

1. **Compute queries, keys, and values.** Apply the three learned linear transformations to the input:

   $$ Q=X~\Omega_Q,\qquad K=X~\Omega_K,\qquad V=K~\Omega_V $$

2. **Compute attention scores.** Calculate the dot product between every query and every key, then scale the scores by the square root of the query dimension:

   $$ S=\frac{QK^T}{\sqrt{d_k}} $$

   The resulting tensor should have shape `(batch_size, seq_len, seq_len)`.

3. **Apply the padding mask (if provided).** The optional `padding_mask` has shape `(batch_size, seq_len)`, where `True` indicates a padding token. Set attention scores corresponding to padding keys to negative infinity (`-inf`) so that those positions receive zero attention weight after softmax. Make sure the mask broadcasts correctly across all query positions.

4. **Compute attention weights.** Apply softmax along the last dimension of the score tensor:

   $$ A=\operatorname{softmax}(S) $$

5. **Compute contextualized embeddings.** Multiply the attention weights by the value vectors:

   $$ Z=AV $$

   Return the resulting tensor, which should have the same shape as the input.



In [ ]:

import torch
import torch.nn as nn
import torch.nn.functional as F

class SingleHeadAttention(nn.Module):
    def __init__(self, d_model=128):
        super().__init__()

        pass

    def forward(self, x, padding_mask=None):
        pass


**Test Your Implementation**

Create an instance of `SingleHeadAttention` with `d_model=128` and pass a randomly generated tensor of shape `(32, 64, 128)` through it.

Verify that the output has shape `(32, 64, 128)`.

Next, create a padding mask that marks the final five positions of every sequence as padding. Verify that your implementation still produces an output of the expected shape and contains no `NaN` values.

You may assume that every sequence contains at least one non-padding token.

<!-- **Questions to Consider**

1. Why do we use three separate learnable transformations for queries, keys, and values instead of using the input embeddings directly?
2. Why do we divide attention scores by \(\sqrt{d_k}\) before applying softmax?
3. Why is softmax applied along the last dimension of the attention score tensor?
4. Why do we replace scores corresponding to padding keys with negative infinity?
5. Why does self-attention produce an output with the same shape as its input in this implementation?

**Hint:** You may find the following PyTorch operations useful: `nn.Linear`, `transpose`, `@` (matrix multiplication), `masked_fill`, and `F.softmax`. -->

# Q4. Transformer Encoder Layer

In this exercise, you will build a **Transformer Encoder layer** using the `SingleHeadAttention` class you implemented in the previous exercise.

A Transformer Encoder combines self-attention with a feed-forward neural network. It also uses **residual connections** and **layer normalization** to help stabilize training.

<br/>

Create the `TransformerEncoder` Class

Create a class called `TransformerEncoder` that inherits from `nn.Module`.

**1. `__init__(self, d_model=128)`**

Initialize the following components:

- **Self-attention:** Create an instance of your `SingleHeadAttention` class with embedding dimension `d_model`.
- **Feed-forward network:** Use `nn.Sequential` to construct a neural network consisting of:
  - A linear layer that expands the embedding dimension from `d_model` to `4 * d_model`.
  - A ReLU activation function.
  - A second linear layer that projects the representation back to `d_model` dimensions.
- **Layer normalization:** Create two separate `nn.LayerNorm` layers, each operating on `d_model` dimensions.

**2. `forward(self, x, padding_mask=None)`**

The input `x` has shape `(batch_size, seq_len, d_model)`. Implement the following steps:

1. **Self-attention with residual connection.**
   - Apply the first layer normalization to the input.
   - Pass the normalized input through your `SingleHeadAttention` module, supplying the optional padding mask.
   - Add the attention output to the original input using a residual connection.

   $$ X' = X + \operatorname{Attention}(\operatorname{LayerNorm}_1(X)) $$

2. **Feed-forward network with residual connection.**
   - Apply the second layer normalization to the output of the previous step.
   - Pass the normalized representation through the feed-forward network.
   - Add the feed-forward output to the representation from the previous step.

  $$ Z = X' + \operatorname{FFN}(\operatorname{LayerNorm}_2(X')) $$

3. Return the final tensor `Z`.

This arrangement is called a **Pre-LayerNorm Transformer Encoder**, because layer normalization is applied before the attention and feed-forward operations.



In [ ]:

class TransformerEncoder(nn.Module):
    def __init__(self, d_model=128):
        super().__init__()
        pass

    def forward(self, x, padding_mask=None):
        pass


**Test Your Implementation**

Create an instance of `TransformerEncoder` with `d_model=128`.

Generate a random input tensor with shape `(32, 64, 128)` and pass it through the encoder.

Verify that:

- The output has shape `(32, 64, 128)`.
- The output contains no `NaN` values.
- The encoder accepts an optional padding mask with shape `(32, 64)`.

<!-- **Questions to Consider**

1. What is the purpose of the feed-forward network after self-attention? How does its operation differ from self-attention?
2. Why does the feed-forward network first expand the embedding dimension and then project it back to its original size?
3. What is the purpose of the two residual connections?
4. What does `nn.LayerNorm(d_model)` normalize, and why is normalization useful?
5. Why must the self-attention and feed-forward modules produce outputs with the same dimensions as their respective inputs?

**Hint:** You may find `nn.Sequential`, `nn.Linear`, `nn.ReLU`, and `nn.LayerNorm` useful. Remember that a residual connection is implemented by adding a module's output to its input. -->

# Q5. Building a Masked Language Model

In this exercise, you will combine the components developed in the previous exercises to build a complete **Masked Language Model (MLM)** using PyTorch.

Your model will use token embeddings, positional embeddings, and two Transformer Encoder layers to generate contextualized representations of input tokens. A final linear layer will produce scores for every token in the vocabulary, allowing the model to predict the original tokens at masked positions.

<br/>

Create the `MaskedLanguageModel` Class

Create a class called `MaskedLanguageModel` that inherits from `nn.Module`.

**1. `__init__(self)`**

Initialize the following components:

- **Token embeddings:** Create an `nn.Embedding` layer that maps each token ID in the vocabulary (`VOCAB_SIZE`) to a 128-dimensional embedding.
- **Positional embeddings:** Create another `nn.Embedding` layer that represents each position in a sequence of length `SEQ_LEN` using a 128-dimensional vector.
- **Transformer Encoders:** Use `nn.ModuleList` to create a stack of **two** instances of your previously implemented `TransformerEncoder` class, each with `d_model=128`.
- **Final layer normalization:** Create an `nn.LayerNorm` layer operating on 128-dimensional vectors.
- **Output layer:** Create a linear layer that maps each 128-dimensional contextualized representation to `VOCAB_SIZE` output scores.

**2. `forward(self, x)`**

The input `x` is a tensor of token IDs with shape `(batch_size, seq_len)`. Implement the following steps:

1. **Generate position indices.** Create a tensor containing the position indices `0, 1, ..., seq_len - 1`. Ensure that this tensor is on the same device as the input.

2. **Compute input embeddings.** Look up the token embeddings and positional embeddings, then add them together:

   $$ H = \operatorname{TokenEmbedding}(X) + \operatorname{PositionEmbedding}(P) $$

   The resulting tensor should have shape `(batch_size, seq_len, 128)`.

3. **Create a padding mask.** Identify positions where the input token ID equals `PAD_ID`. The resulting Boolean mask should have shape `(batch_size, seq_len)`.

4. **Apply the Transformer Encoders.** Pass the embeddings sequentially through both Transformer Encoder layers, supplying the padding mask to each encoder. The output of the first encoder becomes the input to the second.

5. **Apply final layer normalization.** Normalize the output of the final Transformer Encoder using the additional `nn.LayerNorm` layer.

6. **Generate vocabulary scores.** Pass the normalized representations through the output linear layer.

   The final output should have shape:

   $$ (\text{batch size},\text{seq len},\text{VOCAB\SIZE}) $$

   Each position contains a score (logit) for every token in the vocabulary. Return these logits without applying softmax.



In [ ]:

class MaskedLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        pass

    def forward(self, x):
        pass


model = MaskedLanguageModel().to(device)


Create an instance of `MaskedLanguageModel` and move it to the device specified by the `device` variable.

Retrieve one batch of masked input sequences from the `train_loader` created earlier and pass it through the model.

Verify that:

- The input has shape `(BATCH_SIZE, SEQ_LEN)` for a full batch.
- The output has shape `(BATCH_SIZE, SEQ_LEN, VOCAB_SIZE)`.
- The output contains no `NaN` values.


# Q6. Training and Evaluating the Masked Language Model

In this exercise, you will implement the **training and validation loops** for the Masked Language Model (MLM) developed in the previous exercises.

Your model will learn to predict the original tokens at masked positions by minimizing the cross-entropy loss between its predictions and the correct token IDs. You will also evaluate the model on a separate validation dataset to monitor its ability to generalize to unseen text.

### 1. Configure the Optimizer and Loss Function

Initialize the following components:

- **Optimizer:** Use `torch.optim.AdamW` with the model's parameters and a learning rate of `1e-3`.
- **Loss function:** Use `nn.CrossEntropyLoss` with `ignore_index=-100`.
- **Number of epochs:** Set `num_epochs = 20`.

Recall that your `MaskedDataset` assigns a label of `-100` to every unmasked position. By setting `ignore_index=-100`, the loss function will ignore these positions and calculate the loss only for tokens selected for masking.

### 2: Implement the Training Loop

Train the model for 20 epochs. During each epoch, perform the following steps:

1. **Enable training mode.** Call `model.train()` and initialize a variable to accumulate the training loss.

2. **Iterate through the training DataLoader.** For each batch, retrieve the masked input sequences (`x`) and their corresponding labels. Move both tensors to the appropriate device.

3. **Reset gradients.** Use the optimizer to clear gradients accumulated during the previous iteration.

4. **Perform a forward pass.** Pass the masked input sequences through the model to obtain logits with shape:

   $$
   (\text{batch size},\text{seq len},\text{VOCAB\_SIZE})
   $$

5. **Calculate the loss.** Reshape the logits into a two-dimensional tensor of shape `(-1, VOCAB_SIZE)` and flatten the labels into a one-dimensional tensor. Pass both to the cross-entropy loss function.

6. **Perform backpropagation.** Compute the gradients of the loss with respect to the model's learnable parameters.

7. **Apply gradient clipping.** Use `torch.nn.utils.clip_grad_norm_` to limit the global gradient norm to `1.0`. This helps prevent excessively large gradients from destabilizing training.

8. **Update the parameters.** Use the optimizer to update the model's weights.

9. **Accumulate the loss.** Add the current batch's loss to the total training loss for the epoch.

### 3. Implement the Validation Loop

After completing the training batches for each epoch, evaluate the model using `val_loader`.

1. Switch the model to evaluation mode using `model.eval()`.
2. Initialize a variable to accumulate validation loss.
3. Disable gradient computation using `torch.no_grad()`.
4. Iterate through the validation DataLoader, moving inputs and labels to the appropriate device.
5. Perform a forward pass and calculate the cross-entropy loss using the same reshaping operations as in training.
6. Accumulate the validation loss across batches.

**Important:** Do not perform backpropagation or update the model's parameters during validation.

### 4. Monitor Training Progress

At the end of each epoch, calculate the average training and validation losses by dividing their accumulated values by the number of batches in the corresponding DataLoader.

Print the epoch number, average training loss, and average validation loss, formatted to four decimal places.

Your output should follow this general format:

```text
Epoch 1 | Train Loss: ... | Val Loss: ...
Epoch 2 | Train Loss: ... | Val Loss: ...
...
Epoch 20 | Train Loss: ... | Val Loss: ...
```

Observe how both losses change over the 20 epochs. Ideally, the training and validation losses should decrease as the model learns to predict masked tokens.


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss(ignore_index=-100)

num_epochs = 20

for epoch in range(num_epochs):
    model.train()
    train_loss = 0

    for x, labels in train_loader:
        x = x.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        logits = model(x)

        loss = criterion(
            logits.reshape(-1, VOCAB_SIZE),
            labels.reshape(-1)
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        train_loss += loss.item()

    model.eval()
    val_loss = 0

    with torch.no_grad():
        for x, labels in val_loader:
            x = x.to(device)
            labels = labels.to(device)

            logits = model(x)

            loss = criterion(
                logits.reshape(-1, VOCAB_SIZE),
                labels.reshape(-1)
            )

            val_loss += loss.item()

    print(
        f"Epoch {epoch+1} | "
        f"Train Loss: {train_loss/len(train_loader):.4f} | "
        f"Val Loss: {val_loss/len(val_loader):.4f}"
    )

Epoch 1 | Train Loss: 5.9099 | Val Loss: 5.8442
Epoch 2 | Train Loss: 5.6691 | Val Loss: 4.6911
Epoch 3 | Train Loss: 3.7640 | Val Loss: 3.2353
Epoch 4 | Train Loss: 2.9387 | Val Loss: 2.8041
Epoch 5 | Train Loss: 2.6487 | Val Loss: 2.6081
Epoch 6 | Train Loss: 2.4814 | Val Loss: 2.4689
Epoch 7 | Train Loss: 2.3659 | Val Loss: 2.3735
Epoch 8 | Train Loss: 2.2823 | Val Loss: 2.3047
Epoch 9 | Train Loss: 2.2124 | Val Loss: 2.2435
Epoch 10 | Train Loss: 2.1532 | Val Loss: 2.1860
Epoch 11 | Train Loss: 2.1044 | Val Loss: 2.1489
Epoch 12 | Train Loss: 2.0630 | Val Loss: 2.1148
Epoch 13 | Train Loss: 2.0270 | Val Loss: 2.0782
Epoch 14 | Train Loss: 1.9967 | Val Loss: 2.0496
Epoch 15 | Train Loss: 1.9684 | Val Loss: 2.0320
Epoch 16 | Train Loss: 1.9426 | Val Loss: 2.0059
Epoch 17 | Train Loss: 1.9187 | Val Loss: 1.9789
Epoch 18 | Train Loss: 1.8937 | Val Loss: 1.9608
Epoch 19 | Train Loss: 1.8764 | Val Loss: 1.9409
Epoch 20 | Train Loss: 1.8547 | Val Loss: 1.9279


# Q7. Predict missing tokens in sentences

Since GPT-2's tokenizer does not recognize [MASK] natively, we split the input at each literal [MASK] and insert our special mask ID manually.

In [ ]:
def predict(sentence, top_k=5):
    model.eval()

    parts = sentence.split("[MASK]")
    ids = []
    for i, part in enumerate(parts):
        before_mask = i < len(parts) - 1
        if before_mask:
            part = part.rstrip()   # GPT-2 attaches the space to the next token
        ids.extend(encode(part))
        if before_mask:
            ids.append(MASK_ID)

    if MASK_ID not in ids:
        print("No [MASK] found.")
        return
    if len(ids) > SEQ_LEN:
        raise ValueError("Input is too long.")

    x = torch.tensor([ids], device=device)
    with torch.no_grad():
        logits = model(x)[0]

    print(f"\n{sentence}")
    for pos, token_id in enumerate(ids):
        if token_id != MASK_ID:
            continue

        scores = logits[pos].clone()
        scores[[MASK_ID, PAD_ID, UNK_ID]] = float("-inf")

        probs = torch.softmax(scores, dim=-1)
        values, indices = probs.topk(top_k)

        for value, index in zip(values, indices):
            print(f"  {decode([index.item()])!r:12s} {value.item():.2%}")


  predict("Lily went to the [MASK] to play with her friends.")
  predict("Tom was very [MASK] because he lost his toy.")
  predict("The cat drank some [MASK] from a bowl.")
  predict("Mom gave Ben a big [MASK] and said goodnight.")
  predict("The little bird could not [MASK] because its wing was hurt.")